Đây là script chạy huấn luyện và đánh giá mô hình cơ sở gồm các bước:
- Clone repository từ GitHub
- Load dữ liệu đề bài của cuộc thi 
- Huấn luyện mô hình cơ sở
- Lưu mô hình đã huấn luyện và in ra các chỉ số đánh giá trên tập validation

**Lưu ý**:
- Script này **không thể** chạy local do thời gian chạy lớn và ngốn bộ nhớ. Script bắt buộc phải chạy trên GPU [Kaggle](./HOW_TO_RUN_KAGGLE.md).
- Notebook chạy chung cho **7 mô hình cơ sở** khác nhau của dự án tagflow
- Thời gian chạy cụ thể:
    - Baseline RNN, GRU, LSTM: khoảng 5 đến 10 phút
    - Baseline Random Forest, XGBoost, LightGBM: khoảng 30 đến 40 phút
    - Baseline SVM: dao động từ từ 6 đến 8 tiếng (không khuyến khích chạy baseline này)
- Với những trường hợp có thời gian chạy quá lớn (>30 phút), chúng tôi khuyến khích nên chạy trên Kaggle do có cơ chế chạy offline. Có thể xem chi tiết kết quả chạy của basline RNN, GRU, LSTM tại [đây](../model/MODEL_RESULTS.md)
- Cell code 7 là cell cần phải chú ý, trước khi chạy bạn phải:
    - Thay đổi đường dẫn đến file data (Do nó thay đổi tùy theo tài khoản Kaggle)
    - Chọn loại mô hình cơ sở muốn chạy 

In [1]:
!git clone https://github.com/CryAndRRich/tagflow.git

Cloning into 'tagflow'...
remote: Enumerating objects: 260, done.
remote: Counting objects: 100% (26/26), done.
remote: Compressing objects: 100% (19/19), done.
remote: Total 260 (delta 9), reused 21 (delta 5), pack-reused 234 (from 1)
Receiving objects: 100% (260/260), 72.52 MiB | 37.75 MiB/s, done.
Resolving deltas: 100% (123/123), done.


In [2]:
%cd /kaggle/working/tagflow
TAGFLOW_PATH = "/kaggle/working/tagflow"

/kaggle/working/tagflow


In [3]:
import sys

if TAGFLOW_PATH not in sys.path:
    sys.path.append(TAGFLOW_PATH)

In [4]:
from copy import deepcopy
import numpy as np
import pandas as pd
import torch

In [5]:
from config import *
from preprocess import *
from model import *
from utils import *

In [6]:
RANDOM_SEED = 42
set_seed(RANDOM_SEED)

data_generator = torch.Generator()
data_generator.manual_seed(RANDOM_SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

Random seed set to 42
Device: cuda


In [7]:
INPUT_ROOT = "YOUR INPUT ROOT PATH"
WORK_DIR = "/kaggle/working"

# MODEL_NAME có thể nhận các giá trị là: 
# - baseline_rnn, baseline_lstm, baseline_gru
# - baseline_xgboost, baseline_lightgbm, baseline_randomforest, baseline_svm
MODEL_NAME = "baseline_svm"
map_ext = {
    "baseline_rnn": "pth",
    "baseline_lstm": "pth",
    "baseline_gru": "pth",
    "baseline_xgboost": "joblib",
    "baseline_lightgbm": "joblib",
    "baseline_randomforest": "joblib",
    "baseline_svm": "joblib"
}
EXT = map_ext[MODEL_NAME]
CHECKPOINT_FILE = f"{WORK_DIR}/{MODEL_NAME}_best_model.{EXT}"

In [8]:
data_manager = DataManager(
    input_root = INPUT_ROOT, 
    work_dir = WORK_DIR, 
    config_data = CONFIG_DATA,
    seed_worker = seed_worker,
    data_generator = data_generator,
    random_seed = RANDOM_SEED
)

In [9]:
x_train, y_train, x_val, y_val, x_test = data_manager.get_data()
train_loader, val_loader, test_loader = data_manager.get_dataloaders()

In [10]:
if MODEL_NAME in ["baseline_rnn", "baseline_lstm", "baseline_gru"]:
    loss_fns = get_loss_functions(
        y_df=y_train,
        attribute_cols=data_manager.ATTRIBUTE_COLS,
        num_classes_list=data_manager.NUM_CLASSES_LIST,
        label_smoothing=CONFIG_MODEL.LABEL_SMOOTHING,
        device=DEVICE
    )
    
    scheduler_kwargs = deepcopy(CONFIG_MODEL.SCHEDULER_KWARGS)
    scheduler_kwargs["epochs"] = CONFIG_MODEL.NUM_EPOCHS_STAGE_2
    scheduler_kwargs["steps_per_epoch"] = len(train_loader)
    
    model, optimizer, scheduler = get_model_optim_schedule(
        model_name=MODEL_NAME,
        data=data_manager,
        attribute_idx=None,
        model_kwargs=CONFIG_MODEL.MODEL_KWARGS[MODEL_NAME],
        optim_kwargs=CONFIG_MODEL.OTIMIZER_KWARGS,
        scheduler_kwargs=scheduler_kwargs,
        device=DEVICE
    )
    
    final_model = train_baselines(
        model_name=MODEL_NAME,
        model=model, 
        train_loader=train_loader, 
        val_loader=val_loader, 
        loss_fns=loss_fns, 
        optimizer=optimizer, 
        scheduler=scheduler,
        attribute_list=data_manager.ATTRIBUTE_LIST, 
        num_epochs=10,
        alpha_max_loss=CONFIG_MODEL.ALPHA_MAX_LOSS,
        early_stopping=CONFIG_MODEL.EARLY_STOPPING_STAGE_2,
        checkpoint_file=CHECKPOINT_FILE,
        device=DEVICE,
        verbose=True
    )

In [11]:
if MODEL_NAME in ["baseline_xgboost", "baseline_lightgbm", "baseline_randomforest", "baseline_svm"]:
    model = get_model(
        name=MODEL_NAME, 
        num_classes_list=data_manager.NUM_CLASSES_LIST,
        w2v_tensor=data_manager.W2V_TENSOR,
        random_seed=RANDOM_SEED
    ).to(DEVICE)
    
    final_model = train_baselines(
        model_name=MODEL_NAME, 
        model=model, 
        train_loader=train_loader, 
        checkpoint_file=CHECKPOINT_FILE,
        device=DEVICE
    )

Bắt đầu huấn luyện mô hình SVM...

 Hoàn thành huấn luyện | Mô hình được lưu tại: /kaggle/working/baseline_svm_best_model.joblib


In [12]:
val_predictions = run_inference(
    model=final_model, 
    loader=val_loader,
    attribute_list=data_manager.ATTRIBUTE_LIST,
    device=DEVICE
)
get_stats(
    val_predictions=val_predictions,
    y_true=y_val,
    attribute_list=data_manager.ATTRIBUTE_LIST,
    attribute_cols=data_manager.ATTRIBUTE_COLS
)

Attribute 1 Val Macro F1: 0.4596
Attribute 2 Val Macro F1: 0.1027
Attribute 3 Val Macro F1: 0.0313
Attribute 4 Val Macro F1: 0.4405
Attribute 5 Val Macro F1: 0.0931
Attribute 6 Val Macro F1: 0.0328
Exact-Match Val Accuracy: 0.0001
